## Building A Chatbot
In this notebook We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
groq_api_key = os.getenv('GROQ_API_KEY')

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",  # active model
    groq_api_key=groq_api_key
)

/opt/anaconda3/envs/venvgenai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_core.messages import HumanMessage 
model.invoke([HumanMessage(content="Hi, My name is Kem, i'm a chief AI Engineer")])

AIMessage(content="Nice to meet you, Kem! As a Chief AI Engineer, you must be at the forefront of developing and implementing artificial intelligence solutions. What kind of projects are you currently working on? Are you focused on a specific industry or application of AI? I'm here to listen and learn!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 22, 'total_tokens': 79, 'completion_time': 0.129753113, 'completion_tokens_details': None, 'prompt_time': 0.000122127, 'prompt_tokens_details': None, 'queue_time': 0.018171721, 'total_time': 0.12987524}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_ec4c899792', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda00-2851-7f10-93e2-55240ee1ddff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 57, 'total_tokens': 79})

In [5]:
from langchain_core.messages import AIMessage 
model.invoke(
    [
        HumanMessage(content="Hi, My name is Kem, i'm a chief AI Engineer"),
        AIMessage(content="Nice to meet you, Kem! As a Chief AI Engineer, you must be at the forefront of developing and implementing artificial intelligence solutions. What kind of projects are you currently working on? Are you focused on a specific industry or application of AI? I'm here to listen and learn!"),
        HumanMessage(content="Hey, what's my name and what do i do?")
    ]
)

AIMessage(content='Your name is Kem, and you are a Chief AI Engineer!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 99, 'total_tokens': 113, 'completion_time': 0.031527258, 'completion_tokens_details': None, 'prompt_time': 0.004903393, 'prompt_tokens_details': None, 'queue_time': 0.130956766, 'total_time': 0.036430651}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda05-c96d-7333-aa85-b407b4d27303-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 14, 'total_tokens': 113})

### Message History

You can make your model **stateful** by using a Message History wrapper. This component records every user input and model response, storing them in a data store (memory, database, etc.). When a new request comes in, the previous conversation is retrieved and included in the input, allowing the model to maintain context across interactions.


In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}


def get_session_history(session_id:str)->BaseChatMessageHistory:
    
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
    

with_message_history =RunnableWithMessageHistory(model, get_session_history)

In [7]:
config ={"configurable":{"session_id":"chat1"}}

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Kem, I'm a chief AI Engineer")],
    config=config
)

In [12]:
response.content

"Hello again, Kem! It's great to connect with you. As a Chief AI Engineer, I'm sure you have a deep understanding of the latest advancements in artificial intelligence. What specific areas of AI are you most passionate about, such as natural language processing, computer vision, or reinforcement learning? Or are you more focused on the practical applications of AI in industries like healthcare, finance, or transportation? I'm here to listen and learn from your expertise!"

In [13]:
with_message_history.invoke(
    [HumanMessage(content="What's my name")],
    config=config
)

AIMessage(content="Your name is Kem, and you're a Chief AI Engineer!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 205, 'total_tokens': 218, 'completion_time': 0.029345888, 'completion_tokens_details': None, 'prompt_time': 0.010619097, 'prompt_tokens_details': None, 'queue_time': 0.189900982, 'total_time': 0.039964985}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_ec4c899792', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda1e-f9aa-7d30-b38c-602c5bb0a050-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 13, 'total_tokens': 218})

In [14]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I'm Meta AI. Think of me like an assistant who's here to help you learn, plan, and connect. What can I help you with today?"

In [15]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is Eli")],
    config=config1
)
response.content

"Nice to meet you, Eli! How's your day going so far?"

In [16]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is Eli! We just established that a minute ago.'

### Prompt Templates

Prompt templates are used to transform raw user inputs into a structured format that a language model can understand more effectively. Until now, we’ve only passed simple messages directly to the LLM. Next, we’ll make this more flexible by introducing a system message with specific instructions, while still accepting user messages. After that, we’ll extend the template further by including additional input variables beyond just the conversation text.


In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompts = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an helpful assistant. answer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompts|model

In [18]:
chain.invoke({"messages":[HumanMessage(content="Hi, my name is Kem")]})

AIMessage(content="Hello Kem! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 37, 'total_tokens': 62, 'completion_time': 0.056836583, 'completion_tokens_details': None, 'prompt_time': 0.001467389, 'prompt_tokens_details': None, 'queue_time': 1.509928855, 'total_time': 0.058303972}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_ec4c899792', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda32-2325-76d0-8799-1abb8ce62d0a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 25, 'total_tokens': 62})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain, get_session_history)

In [20]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Kelly")],
    config=config
)

response

AIMessage(content="Hi Kelly! Nice to meet you too! How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 54, 'total_tokens': 72, 'completion_time': 0.041028026, 'completion_tokens_details': None, 'prompt_time': 0.002061116, 'prompt_tokens_details': None, 'queue_time': 0.139894893, 'total_time': 0.043089142}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_ec4c899792', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda36-e5c4-75f1-b969-8ca665d6a441-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72})

In [21]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Kelly! You told me earlier.'

In [22]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [23]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Kem")],"language":"Creole haitian"})
response.content

'Bonjour Kem! Mwen kontan rankontre ou! (Hello Kem! Nice to meet you!) Ki jan ou ye jodi a? (How are you today?)'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [24]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [25]:
config = {"configurable": {"session_id": "chat4"}}
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi, I'm Kem")], "language":"Spanish"},
    config=config
)
response.content


'Hola Kem, ¿cómo estás? ¿En qué puedo ayudarte hoy?'

In [26]:
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="hello, What's my name")], "language":"Spanish"},
    config=config
)
response.content

'¡Hola! Tu nombre es Kem. ¿Cómo estás, Kem?'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [28]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [30]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt
    |model
    )

chain.invoke(
    {
    "messages":messages + [HumanMessage(content="what ice cream do i like?")],
    "language":"English"
    }
)

response.content

'¡Hola! Tu nombre es Kem. ¿Cómo estás, Kem?'

In [31]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

"You asked: what's 2 + 2?"

In [ ]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [ ]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

In [ ]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content